In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold

FRED_API_KEY = "e2f0c957d00e077c2b4a335a206cf2ff"   # ← เปลี่ยนตรงนี้
INPUT_PATH   = "../data/processed/cleaned/cleanedoutput.csv"
OUT_PATH     = "../data/processed/featured/featured_data_final.csv"
LIST_PATH    = "../data/processed/featured/feature_list.txt"
START        = "2015-01-01"
END          = "2026-04-01"


In [2]:
from fredapi import Fred
import yfinance as yf

fred = Fred(api_key=FRED_API_KEY)
print("FRED connection")

FRED connection


# 1 : LOAD RAW DATA

In [3]:
df = pd.read_csv(INPUT_PATH, parse_dates=["Date"], index_col="Date")
df.sort_index(inplace=True)

print(f"  Shape      : {df.shape}")
print(f"  Date range : {df.index.min().date()} → {df.index.max().date()}")
print(f"  Missing    : {df.isnull().sum().sum()}")

bdays = pd.bdate_range(start=df.index.min(), end=df.index.max())


def to_bday_series(raw_series: pd.Series) -> pd.Series:
    """Monthly/Weekly FRED → daily business-day via forward fill (no lookahead)."""
    s = raw_series.copy()
    s.index = pd.to_datetime(s.index)
    return s.reindex(bdays, method="ffill")

  Shape      : (2824, 20)
  Date range : 2015-01-02 → 2026-03-30
  Missing    : 0


# 2 : RAW RETURNS 

In [4]:
df = pd.read_csv(INPUT_PATH, parse_dates=["Date"], index_col="Date")
df.sort_index(inplace=True)

print(f"  Shape      : {df.shape}")
print(f"  Date range : {df.index.min().date()} → {df.index.max().date()}")
print(f"  Missing    : {df.isnull().sum().sum()}")

bdays = pd.bdate_range(start=df.index.min(), end=df.index.max())


def to_bday_series(raw_series: pd.Series) -> pd.Series:
    """Monthly/Weekly FRED → daily business-day via forward fill (no lookahead)."""
    s = raw_series.copy()
    s.index = pd.to_datetime(s.index)
    return s.reindex(bdays, method="ffill")

  Shape      : (2824, 20)
  Date range : 2015-01-02 → 2026-03-30
  Missing    : 0


# 3 : TECHNICAL FEATURES

In [5]:
# --- MA & Distance จาก MA ---
# gold_close และ rolling MA คำนวณ ณ ปิดตลาด t → เป็น feature ของวัน t ✓
for window in [5, 10, 20, 30, 60]:
    df[f"gold_ma{window}"]      = df["gold_close"].rolling(window).mean()
    df[f"gold_dist_ma{window}"] = (
        (df["gold_close"] - df[f"gold_ma{window}"]) /
         df[f"gold_ma{window}"] * 100
    )   # ← ไม่ shift: รู้ได้ ณ ปิดตลาดวัน t

In [6]:
df ["gold_ma_cross_5_20"] = df ["gold_ma5"] - df ["gold_ma20"]

In [7]:
# --- Rate-of-Change ---
for lag in [1, 3, 5, 10]:
    df[f"gold_roc_{lag}d"] = df["gold_close"].pct_change(lag) * 100  # ← ไม่ shift


In [8]:
# --- Price Range / Candle Body ---
# ⚠️ OHLC วันนี้รู้ได้หลัง close เท่านั้น
#    ถ้าทำนาย ณ ต้นวัน t ข้อมูลนี้ยังไม่มี → ต้อง shift(1)
#    แต่ถ้าทำนาย ณ หลังปิดตลาดวัน t → ไม่ต้อง shift
#    ⚡ เลือก: ใช้ .shift(1) เพราะ pipeline นี้ทำนาย t+1 ก่อนเปิดตลาด
df["gold_range_pct"] = (
    (df["gold_high"] - df["gold_low"]) / df["gold_low"] * 100
)#.shift(1)   # ← shift(1): ใช้ OHLC ของเมื่อวาน

df["gold_body_pct"] = (
    (df["gold_close"] - df["gold_open"]) / df["gold_open"] * 100
)#.shift(1)   # ← shift(1): ใช้ OHLC ของเมื่อวาน

In [9]:
# --- Lagged Returns ---
# gold_ret_lag1 ณ วัน t = return วัน t-1 → .shift(1) ถูกต้องโดยนิยาม
for lag in [1, 2, 3, 5]:
    df[f"gold_ret_lag{lag}"]  = df["gold_close_ret"].shift(lag)
    df[f"sp500_ret_lag{lag}"] = df["sp500_close_ret"].shift(lag)

In [10]:
# --- Log Price ---
# log(close_t) รู้ได้ ณ ปิดตลาด t → ใช้เป็น feature ของวัน t ✓
for col in ["gold_close", "sp500_close", "oil_close"]:
    df[f"log_{col}"] = np.log(df[col])   # ← ไม่ shift

# 4 : VOLATILITY & REGIME FEATURES

In [11]:
# --- Rolling Volatility ---
for w in [5, 10, 30, 60]:
    df[f"gold_vol_{w}d"] = (
        df["gold_close_ret"].rolling(w).std() * np.sqrt(252) * 100
    )   # ← ไม่ shift: rolling ใช้ข้อมูลถึง t


In [12]:
for w in [7, 30]:
    df[f"sp500_vol_{w}d"] = (
        df["sp500_close_ret"].rolling(w).std() * np.sqrt(252) * 100
    )   # ← ไม่ shift


In [13]:
for w in [7, 30]:
    df[f"sp500_vol_{w}d"] = df["sp500_close_ret"].rolling(w).std() * np.sqrt(252) * 100


In [14]:
# --- Volatility Spread ---
df["vol_spread_vix_gold"] = df["vix_close"] - df["gold_vol_30d"]  # ← ไม่ shift


In [15]:
# --- VIX Momentum ---
df["vix_mom_3d"] = df["vix_close"].pct_change(3) * 100   # ← ไม่ shift
df["vix_mom_5d"] = df["vix_close"].pct_change(5) * 100   # ← ไม่ shift


In [16]:
# --- High-Vol Regime ---
df["regime_high_vol"] = (df["gold_vol_30d"] > 25).astype(int)   # ← ไม่ shift


In [17]:
# --- Vol Z-Score ---
df["gold_vol_zscore"] = (
    (df["gold_vol_30d"] - df["gold_vol_30d"].rolling(252).mean()) /
     df["gold_vol_30d"].rolling(252).std()
)   # ← ไม่ shift


In [18]:
# --- Log Vol / Log VIX ---
df["log_gold_vol_30d"] = np.log(df["gold_vol_30d"].clip(lower=0.01))  # ← ไม่ shift
df["log_vix"]          = np.log(df["vix_close"])                       # ← ไม่ shift


# 5 : CROSS-ASSET FEATURES

In [19]:
df["gold_sp500_ratio"]    = df["gold_close"] / df["sp500_close"]   # ← ไม่ shift
df["oil_gold_ratio"]      = df["oil_close"]  / df["gold_close"]    # ← ไม่ shift
df["dxy_ret1d"]           = df["dxy_close"].pct_change() * 100     # ← ไม่ shift
df["yield_x_dxy"]         = df["yield_close"] * df["dxy_close"]    # ← ไม่ shift
df["gold_oil_spread_ret"] = df["gold_close_ret"] - df["oil_close_ret"]  # ← ไม่ shift


# 6 : ROLLING CORRELATION

In [20]:
gold_ret = df["gold_close_ret"]

In [21]:
for col, name in [
    ("dxy_close_ret",   "dxy"),
    ("sp500_close_ret", "sp500"),
    ("oil_close_ret",   "oil"),
    ("yield_close_ret", "yield"),
    ("vix_close_ret",   "vix"),
]:
    if col not in df.columns:
        df[col] = df[col.replace("_ret", "")].pct_change() * 100
    df[f"corr_90d_{name}"] = gold_ret.rolling(90).corr(df[col])   # ← ไม่ shift


# 7 : CALENDAR FEATURES

In [ ]:
df["day_of_week"]   = df.index.dayofweek
df["month"]         = df.index.month
df["quarter"]       = df.index.quarter
df["is_month_end"]  = df.index.is_month_end.astype(int)
df["days_gap"]      = df.index.to_series().diff().dt.days.fillna(1).astype(int)
df["rollover_flag"] = df.index.day.isin([26, 27, 28, 29]).astype(int)


# 8 : MACRO REGIME FEATURES — FRED

In [23]:
# ── Fed Funds Rate (publication lag ~15 วัน)
print("   FEDFUNDS...")
fed_raw   = fred.get_series("FEDFUNDS", observation_start=START, observation_end=END)
fed_daily = to_bday_series(fed_raw).shift(15)   # simulate publication lag


   FEDFUNDS...


In [24]:
df["f_fed_rate"]        = fed_daily.reindex(df.index)
df["f_fed_rate_chg_3m"] = fed_daily.diff(63).reindex(df.index)
df["f_fed_hike_cycle"]  = (fed_daily.diff(63) > 0).astype(int).reindex(df.index)


In [25]:
# ── Yield Curve
print("   DGS2 / DGS10...")
y2_daily  = to_bday_series(fred.get_series("DGS2",  observation_start=START, observation_end=END))
y10_daily = to_bday_series(fred.get_series("DGS10", observation_start=START, observation_end=END))
spread    = y10_daily - y2_daily

df["f_yield_curve"]       = spread.reindex(df.index)
df["f_yield_curve_slope"] = spread.diff(20).reindex(df.index)
df["f_inverted_curve"]    = (spread < 0).astype(int).reindex(df.index)


   DGS2 / DGS10...


In [26]:
# ── Real Rate (TIPS 10Y) — daily publication, ใช้ได้ทันที
print("   DFII10...")
tips_daily = to_bday_series(fred.get_series("DFII10", observation_start=START, observation_end=END))

df["f_real_rate_10y"]      = tips_daily.reindex(df.index)
df["f_real_rate_chg_1m"]   = tips_daily.diff(21).reindex(df.index)
df["f_real_rate_negative"]  = (tips_daily < 0).astype(int).reindex(df.index)


   DFII10...


In [27]:
# แก้ observation_start เป็นดึงตั้งแต่ปี 2013-01-01
cpi_raw = fred.get_series("CPIAUCSL", observation_start="2013-01-01", observation_end=END)
cpi_daily = to_bday_series(cpi_raw.pct_change(12) * 100).shift(15)

df["f_cpi_yoy"]        = cpi_daily.reindex(df.index)
df["f_cpi_trend_3m"]   = cpi_daily.diff(63).reindex(df.index)
df["f_high_inflation"] = (cpi_daily > 4).astype(int).reindex(df.index)


# 9 : MARKET STRESS FEATURES

In [28]:
# ── VIX Regime
vix_raw = df["vix_close"]

df["f_vix_regime"]    = pd.cut(
    vix_raw, bins=[0, 15, 25, 35, 9999], labels=[0, 1, 2, 3]
).astype(float)   # ← ไม่ shift

df["f_vix_spike"]     = (vix_raw > vix_raw.rolling(60).mean() * 1.5).astype(int)   # ← ไม่ shift
df["f_vix_trend_20d"] = vix_raw.diff(20)                                            # ← ไม่ shift
df["f_vix_ma_ratio"]  = vix_raw / vix_raw.rolling(252).mean()                      # ← ไม่ shift


In [29]:
# ── DXY Trend (คำนวณก่อน drop dxy_close)
if "dxy_close" in df.columns:
    dxy_raw = df["dxy_close"]
    df["f_dxy_trend_20d"]   = dxy_raw.diff(20)                                  # ← ไม่ shift
    df["f_dxy_trend_60d"]   = dxy_raw.diff(60)                                  # ← ไม่ shift
    df["f_dxy_above_ma200"] = (dxy_raw > dxy_raw.rolling(200).mean()).astype(int)  # ← ไม่ shift
else:
    print("   WARNING: dxy_close ไม่พบ ข้าม DXY trend features")
    df["f_dxy_trend_20d"]   = np.nan
    df["f_dxy_trend_60d"]   = np.nan
    df["f_dxy_above_ma200"] = np.nan


In [30]:
# # ── HY Spread
# print("   BAMLH0A0HYM2EY...")
# hy_daily = to_bday_series(
#     fred.get_series("BAMLH0A0HYM2EY", observation_start=START, observation_end=END)
# )

# df["f_hy_spread"]         = hy_daily.reindex(df.index)                                      # ← ไม่ shift
# df["f_hy_spread_chg_20d"] = hy_daily.diff(20).reindex(df.index)                             # ← ไม่ shift
# df["f_credit_stress"]     = (
#     hy_daily > hy_daily.rolling(252).quantile(0.75)
# ).astype(int).reindex(df.index)   # ← ไม่ shift


# 10 : GOLD MICROSTRUCTURE

In [31]:
# Calendar seasonality — deterministic ✓
df["f_month_sin"] = np.sin(2 * np.pi * df.index.month / 12)
df["f_month_cos"] = np.cos(2 * np.pi * df.index.month / 12)
df["f_is_q1"]     = df.index.month.isin([1, 2, 3]).astype(int)
df["f_is_q4"]     = df.index.month.isin([10, 11, 12]).astype(int)


In [32]:
# Momentum Regime — rolling ณ วัน t → ไม่ shift
ret_1m = df["gold_close_ret"].rolling(21).mean()
ret_3m = df["gold_close_ret"].rolling(63).mean()

df["f_gold_momentum_regime"] = pd.Series(
    np.where(
        (ret_1m > 0) & (ret_3m > 0), 2,
        np.where((ret_1m < 0) & (ret_3m < 0), 0, 1)
    ),
    index=df.index
)   # ← ไม่ shift


In [33]:
# Vol Regime — gold_vol_30d เป็น feature ณ วัน t → rolling median ของมัน ก็ ณ วัน t
gold_vol    = df["gold_vol_30d"]
vol_med     = gold_vol.rolling(252).median()

df["f_gold_vol_regime"] = pd.Series(
    np.where(gold_vol > vol_med * 1.5, 2,
             np.where(gold_vol < vol_med * 0.7, 0, 1)),
    index=df.index
)   # ← ไม่ shift


# 11 : DROP RAW PRICE COLUMNS

In [34]:
df.drop(columns=["gold_vol"], inplace=True, errors="ignore")

raw_price_cols = [
    "gold_close", "gold_high", "gold_low", "gold_open",
    "sp500_close", "dxy_close", "oil_close"
]
df.drop(columns=raw_price_cols, inplace=True, errors="ignore")
df.drop(columns=["gold_return", "sp500_return", "abs_return"],
        inplace=True, errors="ignore")

print("   Done")

   Done


# 12 : NaN FILL FOR MACRO FEATURES

In [35]:
MACRO_FEATURES = [
    "f_fed_rate", "f_fed_rate_chg_3m", "f_fed_hike_cycle",
    "f_yield_curve", "f_yield_curve_slope", "f_inverted_curve",
    "f_real_rate_10y", "f_real_rate_chg_1m", "f_real_rate_negative",
    "f_cpi_yoy", "f_cpi_trend_3m", "f_high_inflation",
    "f_vix_regime", "f_vix_spike", "f_vix_trend_20d", "f_vix_ma_ratio",
    "f_dxy_trend_20d", "f_dxy_trend_60d", "f_dxy_above_ma200",
    "f_hy_spread", "f_hy_spread_chg_20d", "f_credit_stress",
    "f_month_sin", "f_month_cos", "f_is_q1", "f_is_q4",
    "f_gold_momentum_regime", "f_gold_vol_regime",
]

In [36]:
existing_macro = [c for c in MACRO_FEATURES if c in df.columns]
df[existing_macro] = df[existing_macro].ffill()   # ffill เท่านั้น (ไม่ bfill)
print(f"   NaN เหลือหลัง ffill: {df[existing_macro].isnull().sum().sum()}")


   NaN เหลือหลัง ffill: 568


# 13 : DEFINE FEATURE LIST

In [37]:
ALL_FEATURES = [
    # ── Returns (วัน t) ──────────────────────────────────────────────────
    "gold_close_ret", "dxy_close_ret", "vix_close_ret",
    "yield_close_ret", "sp500_close_ret", "oil_close_ret",

    # ── Lagged Returns (t-1, t-2, ...) ──────────────────────────────────
    "gold_ret_lag1", "gold_ret_lag2", "gold_ret_lag3", "gold_ret_lag5",
    "sp500_ret_lag1", "sp500_ret_lag3",

    # ── Technical (คำนวณ ณ ปิดตลาด t) ──────────────────────────────────
    "gold_dist_ma5", "gold_dist_ma10", "gold_dist_ma20", "gold_dist_ma30",
    "gold_ma_cross_5_20",
    "gold_roc_1d", "gold_roc_3d", "gold_roc_5d", "gold_roc_10d",
    "gold_range_pct", "gold_body_pct",    # ← shift(1) แล้ว (OHLC เมื่อวาน)

    # ── Volatility & Regime ──────────────────────────────────────────────
    "gold_vol_5d", "gold_vol_10d", "gold_vol_30d", "gold_vol_60d",
    "sp500_vol_7d", "sp500_vol_30d",
    "vol_spread_vix_gold",
    "vix_mom_3d", "vix_mom_5d",
    "regime_high_vol",
    "gold_vol_zscore",
    "log_gold_vol_30d", "log_vix",

    # ── Cross-Asset ──────────────────────────────────────────────────────
    "gold_sp500_ratio", "oil_gold_ratio",
    "dxy_ret1d", "yield_x_dxy",
    "gold_oil_spread_ret",

    # ── Rolling Correlation ──────────────────────────────────────────────
    "corr_90d_dxy", "corr_90d_sp500", "corr_90d_oil",
    "corr_90d_yield", "corr_90d_vix",

    # ── Calendar ─────────────────────────────────────────────────────────
    "day_of_week", "month", "quarter",
    "is_month_end", "days_gap", "rollover_flag",

    # ── Log Price Level ──────────────────────────────────────────────────
    "log_gold_close", "log_sp500_close",

    # ── Stationary Levels ────────────────────────────────────────────────
    "vix_close", "yield_close",

    # ── Macro Regime (FRED, มี publication lag) ──────────────────────────
    "f_fed_rate", "f_fed_rate_chg_3m", "f_fed_hike_cycle",
    "f_yield_curve", "f_yield_curve_slope", "f_inverted_curve",
    "f_real_rate_10y", "f_real_rate_chg_1m", "f_real_rate_negative",
    "f_cpi_yoy", "f_cpi_trend_3m", "f_high_inflation",

    # ── Market Stress ────────────────────────────────────────────────────
    "f_vix_regime", "f_vix_spike", "f_vix_trend_20d", "f_vix_ma_ratio",
    "f_dxy_trend_20d", "f_dxy_trend_60d", "f_dxy_above_ma200",
    "f_hy_spread", "f_hy_spread_chg_20d", "f_credit_stress",

    # ── Gold Microstructure ──────────────────────────────────────────────
    "f_month_sin", "f_month_cos", "f_is_q1", "f_is_q4",
    "f_gold_momentum_regime", "f_gold_vol_regime",
]

In [38]:
# ตรวจ missing
available = [f for f in ALL_FEATURES if f in df.columns]
missing   = [f for f in ALL_FEATURES if f not in df.columns]
if missing:
    print(f"   ⚠ Missing features: {missing}")
else:
    print(f"   ✓ All {len(ALL_FEATURES)} features available")



   ⚠ Missing features: ['f_hy_spread', 'f_hy_spread_chg_20d', 'f_credit_stress']


# 14 : CREATE TARGET & FINALIZE

In [39]:
def create_model_dataset(df: pd.DataFrame,
                         feature_list: list,
                         target_col: str = "gold_close_ret") -> tuple:
    """
    สร้าง dataset สำหรับโมเดล

    ตาราง timeline:
      แถว t │ features = ข้อมูล ≤ ปิดตลาด t   │ target = return ของ t+1

    ขั้นตอน:
      1. shift target -1  (return วันถัดไป)
      2. prefix 'f_' ให้ feature ทุกตัว (ไม่ shift ซ้ำ)
      3. dropna แถว warmup ต้น + แถวสุดท้าย (target = NaN)
    """
    df_m = df.copy()

    # ── TARGET ──────────────────────────────────────────────────────────
    # return วัน t+1 → shift(-1)
    df_m["target_return"] = df_m[target_col].shift(-1)

    # direction: dynamic threshold จาก volatility วัน t (รู้แล้ว ณ ปิดตลาด)
    threshold = df_m["gold_vol_30d"] / np.sqrt(252) * 100 * 0.5
    df_m["target_direction"] = np.where(
        df_m["target_return"] >  threshold,  1,
        np.where(df_m["target_return"] < -threshold, -1, 0)
    )

    # ── FEATURES (prefix f_) ─────────────────────────────────────────────
    feature_cols_out = []
    for f in feature_list:
        if f not in df_m.columns:
            continue
        new_col = f if f.startswith("f_") else f"f_{f}"
        df_m[new_col] = df_m[f]   # ← COPY ตรง ๆ ไม่ shift ซ้ำ
        feature_cols_out.append(new_col)
    # แทรกบรรทัดนี้ก่อนคำสั่ง df_m.dropna() ในฟังก์ชัน create_model_dataset
    print("\n--- เช็ค NaN ก่อน Drop ---")
    nan_counts = df_m[feature_cols_out].isnull().sum()
    print(nan_counts[nan_counts > 100].sort_values(ascending=False).head(10))

    # ── DROPNA ──────────────────────────────────────────────────────────
    drop_subset = feature_cols_out + ["target_return"]
    df_m.dropna(subset=drop_subset, inplace=True)

    return df_m, feature_cols_out


In [40]:
df_model, feature_cols = create_model_dataset(df, available)

print(f"   Shape หลัง dropna : {df_model.shape}")
print(f"   จำนวน Features    : {len(feature_cols)}")
print(f"   Date range        : {df_model.index.min().date()} → {df_model.index.max().date()}")

# ── SANITY CHECK: ยืนยัน timeline ──────────────────────────────────────
print("\n   Sanity check (3 แถวตัวอย่าง):")
sample = df_model[["f_gold_close_ret", "target_return"]].head(3)
print(sample.to_string())
print("   f_gold_close_ret[t] = return ปิดตลาดวัน t")
print("   target_return[t]    = return ปิดตลาดวัน t+1  ✓")



--- เช็ค NaN ก่อน Drop ---
f_gold_vol_zscore    280
f_vix_ma_ratio       251
dtype: int64
   Shape หลัง dropna : (2543, 148)
   จำนวน Features    : 81
   Date range        : 2016-02-12 → 2026-03-27

   Sanity check (3 แถวตัวอย่าง):
            f_gold_close_ret  target_return
Date                                       
2016-02-12         -0.705189       0.000000
2016-02-16          0.000000       0.264919
2016-02-17          0.264919       1.238543
   f_gold_close_ret[t] = return ปิดตลาดวัน t
   target_return[t]    = return ปิดตลาดวัน t+1  ✓


# SAVE

In [ ]:
output_cols = feature_cols + ["target_return", "target_direction"]
df_model[output_cols].to_csv(OUT_PATH)
print(f"   Saved -> {OUT_PATH}")

with open(LIST_PATH, "w") as fh:
    for col in feature_cols:
        fh.write(col + "\n")
print(f"   Saved -> {LIST_PATH}")

   Saved → ../data/processed/featured/featured_data_final.csv
   Saved → ../data/processed/featured/feature_list.txt
